In [3]:
import torch
import torch.nn.functional as F

def beam_search_decoder(data, k, alpha=0.7):
    """
    data: A list of log-probability distributions for each time step (T, V)
    k: Beam width
    alpha: Length penalty coefficient
    """
    # Sequences start with an empty list and a score of 0
    # Format: [sequence, cumulative_log_prob]
    sequences = [([ ], 0.0)]

    # Walk through each time step (T)
    for row in data:
        all_candidates = []

        # Expand each current candidate
        for i in range(len(sequences)):
            seq, score = sequences[i]

            # For each token in vocabulary (V)
            for j in range(len(row)):
                candidate = [seq + [j], score + row[j]]
                all_candidates.append(candidate)

        # Sort all candidates by cumulative log-probability
        ordered = sorted(all_candidates, key=lambda x: x[1], reverse=True)

        # Prune: Keep only top k
        sequences = ordered[:k]

    # Apply Length Penalty to final candidates
    final_results = []
    for seq, score in sequences:
        lp = ((5 + len(seq))**alpha) / ((5 + 1)**alpha)
        normalized_score = score / lp
        final_results.append((seq, normalized_score))

    # Sort by normalized score and return the best
    return sorted(final_results, key=lambda x: x[1], reverse=True)

# --- EXAMPLE CASE ---
# Vocabulary: 0: "The", 1: "movie", 2: "was", 3: "moving", 4: "good"
# We have 3 time steps (T=3) and Vocab size (V=5)
# Log-probabilities for each step:
data = [
    [0.1, 0.2, 0.1, 0.1, 0.1], # Step 1 probabilities
    [0.05, 0.1, 0.5, 0.05, 0.2], # Step 2 probabilities
    [0.1, 0.1, 0.1, 0.4, 0.3]  # Step 3 probabilities
]
data = [torch.log(torch.tensor(row)) for row in data] # Convert to log-space

# Run search with Beam Width k=3
result = beam_search_decoder(data, k=3)

print("Top Beam Results (Normalized):")
for res in result:
    print(f"Sequence: {res[0]}, Score: {res[1]:.4f}")

Top Beam Results (Normalized):
Sequence: [1, 2, 3], Score: -2.6318
Sequence: [1, 2, 4], Score: -2.8670
Sequence: [0, 2, 3], Score: -3.1985


In [5]:
import math

# Step 1: Initial scores
# Greedy only takes "I". Beam (k=2) takes both.
beam = [
    {"seq": ["I"], "score": -0.2},
    {"seq": ["The"], "score": -0.5}
]

# Step 2: Expand all possibilities
candidates = [
    # Paths from "I"
    {"seq": ["I", "am"], "score": -0.2 + -1.5}, # -1.7
    {"seq": ["I", "cat"], "score": -0.2 + -2.0}, # -2.2
    # Paths from "The"
    {"seq": ["The", "cat"], "score": -0.5 + -0.1}, # -0.6 <--- WINNER
    {"seq": ["The", "sat"], "score": -0.5 + -0.4}, # -0.9
]

# Greedy Result (k=1): Would have only looked at "I" paths.
greedy_result = candidates[0] # ["I", "am"] with -1.7

# Beam Result (k=2): Sorts all and finds the best overall.
beam_results = sorted(candidates, key=lambda x: x['score'], reverse=True)[:2]

print(f"Greedy Choice: {greedy_result['seq']} (Score: {greedy_result['score']})")
print(f"Beam Best Choice: {beam_results[0]['seq']} (Score: {beam_results[0]['score']})")

Greedy Choice: ['I', 'am'] (Score: -1.7)
Beam Best Choice: ['The', 'cat'] (Score: -0.6)
